<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week3a1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np

# Load original dataset with verified clean sequence values
customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, np.nan],
    'age':     [34, np.nan, 29, 41, np.nan, 52, 38, np.nan, 27, 45, 31, 36],
    'city':    ['Lagos', 'Abuja', None, 'Kano', 'Lagos', None,
                'Ibadan', 'Abuja', 'Lagos', 'Kano', None, 'Lagos'],
    'income':  [250000, np.nan, 180000, np.nan, np.nan, 420000,
                np.nan, 150000, np.nan, 310000, np.nan, 205000],
    'revenue': [1200, 450, 800, 2300, 300, 5100, 640, 275, 980, 3050, 410, 760],
    'churned': ['No', 'Yes', 'No', 'No', 'Yes', 'No',
                'Yes', 'No', 'No', 'No', 'Yes', 'No'],
})

# --- RESTART-AND-RUN-ALL FIX: Capture the raw state immediately ---
customers_raw = customers.copy()

print("--- PART A: DETECT ---")
print("Missing Value Counts:")
print(customers.isna().sum())

print("\nMissing Percentages (%):")
print((customers.isna().mean() * 100).round(1))

print("\nRows with Missing Age:")
print(customers[customers['age'].isna()])




--- PART A: DETECT ---
Missing Value Counts:
customer_id    1
age            3
city           3
income         6
revenue        0
churned        0
dtype: int64

Missing Percentages (%):
customer_id     8.3
age            25.0
city           25.0
income         50.0
revenue         0.0
churned         0.0
dtype: float64

Rows with Missing Age:
   customer_id  age   city    income  revenue churned
1        102.0  NaN  Abuja       NaN      450     Yes
4        105.0  NaN  Lagos       NaN      300     Yes
7        108.0  NaN  Abuja  150000.0      275      No


In [11]:
print("--- PART C: CHALLENGE MISSING VALUE REPORT ---")

missing_report = pd.DataFrame({
    'missing_count': customers_raw.isna().sum(),
    'missing_pct': (customers_raw.isna().mean() * 100).round(1)
}).sort_values(by='missing_count', ascending=False)

display(missing_report)


--- PART C: CHALLENGE MISSING VALUE REPORT ---


,missing_count,missing_pct
income,6,50.0
age,3,25.0
city,3,25.0
customer_id,1,8.3
revenue,0,0.0
churned,0,0.0


In [12]:
print("--- PART B: DROP, FILL, FLAG ---")

# 1. DROP: Remove rows with no customer_id
# Prediction: Count will drop from 12 to 11.
# Comment: customer_id is a primary key; records without it cannot be validated or tracked safely.
customers = customers.dropna(subset=['customer_id']).copy()

# 2. FILL: Impute age and city profiles
# Expected Age Median: 36.0 (The 8 remaining valid ages yield a median of 36.0)
# Justification (Age): Median imputation limits the influence of extreme value skewness on age tracking.
# Justification (City): Filling with 'Unknown' categorizes missing strings cleanly without dropping rows.
customers['age'] = customers['age'].fillna(customers['age'].median())
customers['city'] = customers['city'].fillna('Unknown')

# 3. FLAG: Generate missing income markers
# Prediction: Incomplete financial records often indicate lower platform interaction and higher churn risks.
customers['income_missing'] = customers['income'].isna()

print("\nChurn Rate Comparison by Income Missing Flag (%):")
churn_comparison = (customers['churned'] == 'Yes').groupby(customers['income_missing']).mean() * 100
display(churn_comparison)

# Comment (Income Decision): Keep income alongside this missingness flag column.
# Dropping the column discards data for 54.5% of rows, while dropping rows leaves our sample size too small.


--- PART B: DROP, FILL, FLAG ---

Churn Rate Comparison by Income Missing Flag (%):


income_missing
False     0.000000
True     66.666667
Name: churned, dtype: float64

In [13]:
print("--- PART C: CHALLENGE MISSING VALUE REPORT ---")

# Build report using the uncleaned dataset before any dropping or filling occurs
missing_report = pd.DataFrame({
    'missing_count': customers.isna().sum(),
    'missing_pct': (customers.isna().mean() * 100).round(1)
}).sort_values(by='missing_count', ascending=False)

display(missing_report)


--- PART C: CHALLENGE MISSING VALUE REPORT ---


,missing_count,missing_pct
income,6,54.5
customer_id,0,0.0
age,0,0.0
city,0,0.0
revenue,0,0.0
churned,0,0.0
income_missing,0,0.0
